# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MEDOXIII/FlyRank-Internship-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MEDOXIII/FlyRank-Internship-ML"
REPO_DIR = "FlyRank-Internship-ML"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found"

Working dir: D:\Projects\FlyRank-Internship-ML\FlyRank-Internship-ML


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I build the vector by running the repo's reference pipeline (`scripts/01_prepare_features.py`) rather than reimplementing feature prep myself — `GUIDE.md` calls it the baseline to run and copy, not edit. It fills numeric blanks with 0, categoricals with `"unknown"`, keeps rows with `impressions_90d > 0` and `content_age_days >= 90` (0 rows dropped here since the starter slice already guarantees both — confirmed in the ML-04 contract), and adds the `log1p` / `has_*` / `measurable_opportunity` engineered columns. The model-facing set is exactly `MODEL_NUMERIC_FEATURES` + `MODEL_CATEGORICAL_FEATURES` from `scripts/ml_utils.py` — 26 columns (18 numeric, 8 categorical) — the same "feature" bucket the ML-04 contract already classified. Everything the contract put in "excluded" is simply never selected into `X` below; Section 4 re-verifies that.

In [2]:
import subprocess
import sys
from pathlib import Path
import pandas as pd

# Self-contained: correct whether or not the bootstrap cell already chdir'd to the root.
REPO_ROOT = next(c for c in [Path("."), Path(".."), Path("../..")]
                 if (c / "scripts" / "ml_utils.py").exists()).resolve()
sys.path.insert(0, str(REPO_ROOT / "scripts"))

result = subprocess.run(
    [sys.executable, "scripts/01_prepare_features.py"],
    cwd=REPO_ROOT, capture_output=True, text=True,
)
print(result.stdout.strip())
if result.returncode != 0:
    raise RuntimeError(result.stderr)

from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, PROCESSED_DIR, RAW_PATH

fv = pd.read_csv(PROCESSED_DIR / "refresh_feature_vector.csv")
raw = pd.read_csv(RAW_PATH)
print(f"feature vector: {fv.shape[0]:,} rows x {fv.shape[1]} cols (raw was {raw.shape[0]:,} x {raw.shape[1]})")

X = fv[MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES]
y = fv["is_declining_label"]
print("numeric features:", len(MODEL_NUMERIC_FEATURES), "| categorical features:", len(MODEL_CATEGORICAL_FEATURES))
print("label rate:", round(y.mean(), 4))

Prepared 30,000 rows from 30,000 raw rows
Wrote D:\Projects\FlyRank-Internship-ML\FlyRank-Internship-ML\data\processed\refresh_feature_vector.csv


feature vector: 30,000 rows x 52 cols (raw was 30,000 x 44)
numeric features: 18 | categorical features: 8
label rate: 0.5421


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The 26 features split into two families with different "available-when" answers:

- **Static / snapshot properties** (7 numeric + 6 categorical) — `search_volume`, `competition`, `cpc`, `word_count`, `char_count`, `content_age_days`, `days_since_last_update`, `competition_level`, `content_type`, `main_intent`, `age_tier`, `freshness_tier`, `word_count_tier`. These describe the page or its target keyword, not a 90-day activity total — knowable independent of whatever the last-30-day trend turns out to be.
- **90-day-window aggregates** (11 numeric + 2 categorical) — `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, the four `log_*_90d` totals, `days_with_impressions`, `days_with_sessions`, and the two tiers built from them (`impression_tier`, `position_tier`). Every one of these sums or averages over the *same* 90 days that contain the label's last-30-day sub-window (`impressions_last_30d` is a strict subset of `impressions_90d`, proved in ML-04). They're "available" in the sense of already sitting in the export, but they are not cleanly *before* the trend they're meant to predict — Section 3 measures how much that overlap is worth.

**Missing values:** numeric blanks are filled with 0 by the reference pipeline; categoricals get `"unknown"`. `word_count`/`char_count` were blank for 25.7% of raw rows before that fill (checked below) — the pipeline adds no `has_word_count`-style flag, so a 0-fill is indistinguishable from "genuinely short." That's a residual risk the ML-04 contract already flagged; not mine to patch since `scripts/01_prepare_features.py` is the fixed baseline.

In [3]:
window_num = ["log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
              "days_with_impressions", "days_with_sessions", "ctr", "avg_position",
              "engagement_rate", "scroll_rate", "ai_traffic_pct"]
static_num = ["search_volume", "competition", "cpc", "word_count", "char_count",
              "content_age_days", "days_since_last_update"]
window_cat = ["impression_tier", "position_tier"]
static_cat = ["competition_level", "content_type", "main_intent", "age_tier",
              "freshness_tier", "word_count_tier"]

assert set(window_num + static_num) == set(MODEL_NUMERIC_FEATURES)
assert set(window_cat + static_cat) == set(MODEL_CATEGORICAL_FEATURES)
print("static/snapshot features:", len(static_num) + len(static_cat))
print("90-day-window features:", len(window_num) + len(window_cat))

raw_equivalent = {
    "log_impressions_90d": "impressions_90d",
    "log_clicks_90d": "clicks_90d",
    "log_sessions_90d": "sessions_90d",
    "log_ai_sessions_90d": "ai_sessions_90d",
}
raw_cols_for_missing = [raw_equivalent.get(c, c) for c in MODEL_NUMERIC_FEATURES]
print("\noriginal (pre-fill) missing % for numeric model features, top 8 (log_* mapped to raw source):")
print((raw[raw_cols_for_missing].isna().mean() * 100).round(1).sort_values(ascending=False).head(8))

static/snapshot features: 13
90-day-window features: 13

original (pre-fill) missing % for numeric model features, top 8 (log_* mapped to raw source):
char_count       25.7
word_count       25.7
competition       8.2
search_volume     8.2
cpc               8.2
scroll_rate       0.4
clicks_90d        0.0
sessions_90d      0.0
dtype: float64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Three attacks, run in the order the leakage skill recommends. Base rate is 54.2%, which is the naive benchmark for Precision@50; for AUC the no-skill baseline is 0.5. Every score below sits next to the relevant one.

**1. Label-derived features — add one back on purpose.** The clean 26-feature set scores AUC 0.709 / Precision@50 0.86 on a random split. Adding `trend_pct` back in — a column ML-04 already excluded as the label's direct source — jumps the *same* model to AUC 0.9996 / Precision@50 1.00. That collapse-toward-perfect proves two things at once: `trend_pct` really is leakage, and my test harness is sensitive enough to catch it (the skill's own sanity check for the sanity check).

**2. Overlapping windows.** `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, the `log_*_90d` totals, and the two tiers built from them all aggregate over the full 90 days — which contains the exact last-30-day window the label is defined from. Splitting the clean set in two: window-derived features alone score AUC 0.667; static/snapshot features alone (zero 90-day-window columns) score AUC 0.634. Both land well above the 0.542 base rate, so the ranking signal isn't purely an artifact of the overlap — but the window-derived half is doing real, overlap-tainted work, and the starter CSV has no daily grain to rebuild `ctr`/`avg_position`/etc. from only the pre-label days. I'm keeping them for this directional/decision-support use (per ML-03's framing), but the 0.709 AUC should be read as "same-window correlation," not a clean forecast number — a caveat that belongs in every result this feature set produces.

**3. Split honesty.** A client-grouped split (train/test see disjoint clients, verified 0 overlap) drops the clean model from AUC 0.709 / P@50 0.86 (random split) to AUC 0.611 / P@50 0.74. That ~0.10 AUC / 0.12 precision gap is the model partly memorizing per-client baselines under a random split; the grouped number is the honest one to report.

**Product flags:** not applicable here — `impression_tier`/`position_tier` are transparent threshold buckets of our own numeric features, not a verdict from an external scoring system, so category 3 of the leakage taxonomy doesn't apply to this feature set.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from ml_utils import precision_at_k

def make_pipeline(numcols, catcols):
    pre = ColumnTransformer([
        ("num", StandardScaler(), numcols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), catcols),
    ])
    return Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=1000))])

def score(numcols, catcols, label, groups=None):
    X_ = fv[numcols + catcols]
    if groups is None:
        Xtr, Xte, ytr, yte = train_test_split(X_, y, test_size=0.25, random_state=42, stratify=y)
    else:
        idx_tr, idx_te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(X_, y, groups=groups))
        Xtr, Xte, ytr, yte = X_.iloc[idx_tr], X_.iloc[idx_te], y.iloc[idx_tr], y.iloc[idx_te]
    pipe = make_pipeline(numcols, catcols)
    pipe.fit(Xtr, ytr)
    proba = pipe.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(yte, proba)
    p50 = precision_at_k(yte.values, proba, 50)
    print(f"{label:38s} AUC={auc:.4f}  P@50={p50:.2f}")
    return auc, p50

print(f"base rate (always-positive baseline): {y.mean():.3f}\n")

print("1) label-derived-feature test -- add a known leak back on purpose")
score(MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, "clean feature set")
score(MODEL_NUMERIC_FEATURES + ["trend_pct"], MODEL_CATEGORICAL_FEATURES, "clean set + trend_pct (leak)")

print("\n2) overlapping-window test -- split clean set by window origin")
score(window_num, window_cat, "window-derived only (90d aggregates)")
score(static_num, static_cat, "static/snapshot only (no 90d window)")

print("\n3) split-honesty test -- random vs client-grouped")
score(MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, "clean set, random split")
score(MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, "clean set, client-grouped split", groups=fv["client_id"])

base rate (always-positive baseline): 0.542

1) label-derived-feature test -- add a known leak back on purpose


clean feature set                      AUC=0.7090  P@50=0.86


clean set + trend_pct (leak)           AUC=0.9996  P@50=1.00

2) overlapping-window test -- split clean set by window origin
window-derived only (90d aggregates)   AUC=0.6668  P@50=0.74


static/snapshot only (no 90d window)   AUC=0.6339  P@50=0.82

3) split-honesty test -- random vs client-grouped


clean set, random split                AUC=0.7090  P@50=0.86


clean set, client-grouped split        AUC=0.6111  P@50=0.74


(0.6111010124980224, 0.74)

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- `trend_direction`, `trend_pct` — this *is* the label's source; Section 3's leak-back test shows exactly how much they'd hand the model.
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` — the two numbers `trend_pct`/`trend_direction` are computed from; ML-04 showed they reconstruct the label with 100% accuracy, so they leak it just as hard as `trend_pct` itself.
- `provider_used`, `model_used` — content-generation metadata (which LLM wrote the page); the data dictionary flags both as "not a model feature" since they describe the production pipeline, not the page's search/user-behavior signal.
- `pageviews_90d`, `users_90d`, `engaged_sessions_90d`, `scroll_events_90d` — raw counts already summarized by the rate features I kept (`engagement_rate`, `scroll_rate`) plus the totals above; keeping both would double-count the same signal.
- `char_count_tier`, `age_tier_order` — bucketed/ordinal duplicates of `char_count` and `age_tier`, which are already in the feature set.
- `content_id`, `client_id` — pseudonymous IDs, used only for the client-grouped split above, never as model inputs.

In [5]:
excluded_cols = ["trend_direction", "trend_pct",
                 "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                 "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
                 "provider_used", "model_used",
                 "pageviews_90d", "users_90d", "engaged_sessions_90d", "scroll_events_90d",
                 "char_count_tier", "age_tier_order"]

used_cols = set(MODEL_NUMERIC_FEATURES) | set(MODEL_CATEGORICAL_FEATURES)
leaked_into_X = set(excluded_cols) & used_cols
print("excluded columns declared:", len(excluded_cols))
print("any excluded column present in the model's X:", leaked_into_X if leaked_into_X else "none")
print("content_id / client_id used only for grouping, confirmed absent from X:",
      {"content_id", "client_id"} & used_cols or "none")

excluded columns declared: 16
any excluded column present in the model's X: none
content_id / client_id used only for grouping, confirmed absent from X: none


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.